<a href="https://colab.research.google.com/github/FYasmin12/Flyrank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This notebook is yours to fill. Work the sections **in order**. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read skills/README.md first and load the one skill this assignment names on its card..

In [1]:
import json, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

# Find the repo root (works locally and in Colab; clones the repo if the data is not there)
def find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "data/raw/content_refresh_anonymized.csv").exists():
            return p
    return None

ROOT = find_root()
if ROOT is None:
    if not Path("Flyrank-ML-Internship").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/FYasmin12/Flyrank-ML-Internship.git"], check=True)
    ROOT = Path("Flyrank-ML-Internship").resolve()

raw = pd.read_csv(ROOT / "data/raw/content_refresh_anonymized.csv")
# Same filters as the starter pipeline (scripts/01_prepare_features.py)
df = raw[(raw["impressions_90d"] > 0) & (raw["content_age_days"] >= 90)].drop_duplicates("content_id").copy()

# is_declining_label is used ONLY to check signals and to evaluate. It is never an input to the score.
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
base_rate = df["is_declining_label"].mean()
print(f"pages: {len(df):,} | base rate (share declining): {base_rate:.3f}")

pages: 30,000 | base rate (share declining): 0.542


## 1. My rule and its reason codes

**The rule in plain words:** *A page is worth reviewing first if it is visible (people see it in search), it already ranks on page one, but almost nobody clicks it.* That is the logic behind FlyRank's CTR-fix flag: a page that ranks well but earns few clicks may have a weak title or snippet.

**Reason code:** exactly one, `low_ctr_visible_page`. Pages that do not match get `no_flag`.

My first idea was the usual refresh rule (*stale AND visible*). So before coding anything I check two signals: **staleness** (behind the refresh flags) and **CTR vs position** (behind the CTR-fix flag). Each table prints n and compares to the base rate of 0.542.

### Signal check 1: staleness (behind the refresh flags)

In [2]:
df["stale_bucket"] = pd.cut(df["days_since_last_update"], [0, 30, 90, 180, 1e6], right=False,
                            labels=["<30 days", "30-89 days", "90-179 days", "180+ days"])
t1 = df.groupby("stale_bucket", observed=True)["is_declining_label"].agg(n="count", declining_rate="mean").round(3)
t1["vs_base_rate"] = (t1["declining_rate"] - base_rate).round(3)
print(f"base rate: {base_rate:.3f}")
t1

base rate: 0.542


,n,declining_rate,vs_base_rate
stale_bucket,,,
<30 days,20480,0.511,-0.031
30-89 days,175,0.589,0.047
90-179 days,9171,0.611,0.069
180+ days,174,0.471,-0.071


**Verdict: MIXED.** The idea "older means more likely to decline" only half holds. Pages 90-179 days old decline more often (about 0.61) than pages under 30 days (about 0.51). But the **180+ day bucket falls below the base rate (about 0.47)**, and it has only **174 pages**. So the hard "stale for 180+ days" rule from the refresh flags would flag very few pages and would not pick out declining ones in this data. I do **not** build my rule on a 180-day cutoff. This negative result saved my rule.

### Signal check 2: CTR vs position (behind the CTR-fix flag)

In [3]:
vis = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 10)].copy()
vis["ctr_bucket"] = pd.cut(vis["ctr"], [0, 0.1, 0.5, 1, 1e9], right=False,
                           labels=["<0.1%", "0.1-0.5%", "0.5-1%", "1%+"])
t2 = vis.groupby("ctr_bucket", observed=True)["is_declining_label"].agg(n="count", declining_rate="mean").round(3)
print(f"visible pages ranking in the top 10: {len(vis):,} | base rate in this group: {vis['is_declining_label'].mean():.3f}")
t2

visible pages ranking in the top 10: 7,564 | base rate in this group: 0.589


,n,declining_rate
ctr_bucket,,
<0.1%,1520,0.741
0.1-0.5%,4449,0.586
0.5-1%,1219,0.465
1%+,376,0.410


**Verdict: CONFIRMED (directional).** Among visible pages that rank in the top 10, the lower the CTR the more often the page is declining, in every bucket: about 0.74 for CTR under 0.1%, down to about 0.41 for CTR of 1% or more. Each bucket has hundreds of pages or more. This is an association in observed data, not proof that fixing the title will recover traffic. Also, CTR is a 90-day number that includes the last 30 days, so it overlaps with the label window (see section 4).

## 2. Build the ranked queue (writes the CSV)

**Rule (three sentences):** flag a page if it has at least 500 impressions in 90 days, an average position of 10 or better, and a CTR under 0.5%. Score = `log(1 + impressions)` times how far the CTR sits below 0.5% (1 = no clicks, 0 = at the threshold). Bigger, quieter pages come first. The weights are my own guesses, not fitted to the label.

In [4]:
visible = df["impressions_90d"] >= 500
page_one = (df["avg_position"] > 0) & (df["avg_position"] <= 10)
low_ctr = df["ctr"] < 0.5
flag = visible & page_one & low_ctr

ctr_gap = (0.5 - df["ctr"].clip(upper=0.5)) / 0.5          # 1 = no clicks, 0 = at the 0.5% line
df["score"] = np.where(flag, np.log1p(df["impressions_90d"]) * ctr_gap, 0.0).round(4)
df["reason_code"] = np.where(flag, "low_ctr_visible_page", "no_flag")
df["action"] = np.where(flag, "refresh_and_review_ctr", "monitor")
# Confidence note (built from facts, never from the label): low if the page was edited in the last 30 days,
# is younger than 120 days, or shows 0 clicks (could be a tracking gap). Otherwise medium. Nothing is "high": it is a baseline.
recent = df["days_since_last_update"] <= 30
young = df["content_age_days"] < 120
no_clicks = df["clicks_90d"] == 0
df["confidence"] = np.where(flag, np.where(recent | young | no_clicks, "low", "medium"), "n/a")
df["rank"] = df["score"].rank(method="first", ascending=False).astype(int)

queue = df.sort_values("rank")
cols = ["rank", "content_id", "client_id", "score", "reason_code", "action", "confidence",
        "impressions_90d", "clicks_90d", "avg_position", "ctr", "days_since_last_update", "content_age_days"]
out_dir = ROOT / "work/outputs"
out_dir.mkdir(parents=True, exist_ok=True)
queue[cols].to_csv(out_dir / "baseline_action_score.csv", index=False)

print(f"flagged pages: {int(flag.sum()):,} of {len(df):,}  |  wrote {out_dir / 'baseline_action_score.csv'}")
queue[cols].head(5)

flagged pages: 5,969 of 30,000  |  wrote /content/Flyrank-ML-Internship/work/outputs/baseline_action_score.csv


,rank,content_id,client_id,score,reason_code,action,confidence,impressions_90d,clicks_90d,avg_position,ctr,days_since_last_update,content_age_days
7445,1,content_c8e9d6ab9013,client_19581e27de,12.2486,low_ctr_visible_page,refresh_and_review_ctr,low,208678,0,9.7,0.00,104,362
7678,2,content_8451fc6f034d,client_d029fa3a95,11.7632,low_ctr_visible_page,refresh_and_review_ctr,low,272144,75,2.3,0.03,20,280
27178,3,content_453722754fea,client_f369cb89fc,11.6130,low_ctr_visible_page,refresh_and_review_ctr,low,140079,16,7.6,0.01,20,97
6903,4,content_c84a0ab98e90,client_f369cb89fc,11.5772,low_ctr_visible_page,refresh_and_review_ctr,low,223271,70,7.8,0.03,20,95
3331,5,content_4a6607efcb46,client_6208ef0f77,11.5251,low_ctr_visible_page,refresh_and_review_ctr,medium,128068,17,2.2,0.01,104,148


**How good is it?** Two questions: (a) does the flag itself pick out declining pages, and (b) does the score put the best ones first? precision@K = of the top K pages in my queue, how many were actually declining. The label is used here **only to score the queue**, never to build it. Always read it next to the base rate.

In [5]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return float(np.asarray(labels)[order[:k]].mean())

metrics = {"rows": int(len(df)), "flagged": int(flag.sum()), "base_rate": round(float(base_rate), 3),
           "declining_rate_flagged": round(float(df.loc[flag, "is_declining_label"].mean()), 3),
           "declining_rate_not_flagged": round(float(df.loc[~flag, "is_declining_label"].mean()), 3)}
for k in (10, 20, 50, 200):
    metrics[f"precision_at_{k}"] = round(precision_at_k(df["score"], df["is_declining_label"], k), 3)
(out_dir / "baseline_metrics.json").write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

{
  "rows": 30000,
  "flagged": 5969,
  "base_rate": 0.542,
  "declining_rate_flagged": 0.625,
  "declining_rate_not_flagged": 0.521,
  "precision_at_10": 0.5,
  "precision_at_20": 0.6,
  "precision_at_50": 0.58,
  "precision_at_200": 0.57
}


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
top20 = queue.head(20)[["rank", "content_id", "impressions_90d", "clicks_90d", "avg_position", "ctr",
                         "days_since_last_update", "content_age_days", "score", "reason_code", "action",
                         "confidence", "is_declining_label"]]
top20.reset_index(drop=True)

,rank,content_id,impressions_90d,clicks_90d,avg_position,ctr,days_since_last_update,content_age_days,score,reason_code,action,confidence,is_declining_label
0,1,content_c8e9d6ab9013,208678,0,9.7,0.00,104,362,12.2486,low_ctr_visible_page,refresh_and_review_ctr,low,1
1,2,content_8451fc6f034d,272144,75,2.3,0.03,20,280,11.7632,low_ctr_visible_page,refresh_and_review_ctr,low,0
2,3,content_453722754fea,140079,16,7.6,0.01,20,97,11.6130,low_ctr_visible_page,refresh_and_review_ctr,low,1
3,4,content_c84a0ab98e90,223271,70,7.8,0.03,20,95,11.5772,low_ctr_visible_page,refresh_and_review_ctr,low,0
4,5,content_4a6607efcb46,128068,17,2.2,0.01,104,148,11.5251,low_ctr_visible_page,refresh_and_review_ctr,medium,0
5,6,content_39881853ef0c,112434,10,7.2,0.01,20,97,11.3975,low_ctr_visible_page,refresh_and_review_ctr,low,1
6,7,content_36ff89c8214e,295097,154,7.3,0.05,104,144,11.3356,low_ctr_visible_page,refresh_and_review_ctr,medium,0
7,8,content_0919dd345d80,119217,26,7.0,0.02,7,326,11.2212,low_ctr_visible_page,refresh_and_review_ctr,low,1
8,9,content_c1fe78bc4e37,134055,43,7.5,0.03,104,153,11.0977,low_ctr_visible_page,refresh_and_review_ctr,medium,1
9,10,content_b115f7c74779,123469,37,8.0,0.03,104,313,11.0203,low_ctr_visible_page,refresh_and_review_ctr,medium,0


**Reading the top 20.** All 20 get action `refresh_and_review_ctr` and reason code `low_ctr_visible_page`: each has 38,000 to 295,000 impressions in 90 days, ranks about 2 to 10 on average, and earns almost no clicks (CTR 0.01% to 0.06%). **Confidence** is `low` if the page was edited in the last 30 days, is under 120 days old, or has 0 clicks; otherwise `medium`. No pick is "high", because this is only a baseline. Of the 20, 12 were declining and 8 were not (the same 0.60 as precision@20). One line each, in rank order:

1. **Rank 1** (pos 9.7, 208k impr, **0 clicks**, 104 days since update). Confidence **low**. *Wrong if* the zero clicks are a tracking gap, not a weak snippet.
2. **Rank 2** (pos 2.3, 272k impr, 75 clicks, edited 20 days ago). **Low**. *Wrong if* the query is answered on the results page, so nobody clicks even at position 2.
3. **Rank 3** (pos 7.6, 140k impr, 16 clicks, only 97 days old). **Low**. *Wrong if* the page is too new for a stable position, so the low CTR is temporary.
4. **Rank 4** (pos 7.8, 223k impr, 70 clicks, edited 20 days ago, 95 days old). **Low**. *Wrong if* the recent edit already fixed what a refresh would fix.
5. **Rank 5** (pos 2.2, 128k impr, 17 clicks, 104 days since update). **Medium**. *Wrong if* it ranks 2nd for a broad query where most people leave without clicking.
6. **Rank 6** (pos 7.2, 112k impr, 10 clicks, edited 20 days ago, 97 days old). **Low**. *Wrong if* Google is still settling after the recent edit.
7. **Rank 7** (pos 7.3, 295k impr, 154 clicks, 104 days since update). **Medium**. *Wrong if* average position hides a wide spread (position 1 on one query, 30 on another).
8. **Rank 8** (pos 7.0, 119k impr, 26 clicks, edited **7 days ago**). **Low**. *Wrong if* a refresh is also suggested: it was edited a week ago.
9. **Rank 9** (pos 7.5, 134k impr, 43 clicks, 104 days since update). **Medium**. *Wrong if* a change in the results layout (ads, answer boxes) pulled clicks away, which a better title cannot fix.
10. **Rank 10** (pos 8.0, 123k impr, 37 clicks, 104 days since update). **Medium**. *Wrong if* this is just normal CTR for the topic at position 8 (this page was not declining).
11. **Rank 11** (pos 6.8, 65k impr, 6 clicks, edited 26 days ago). **Low**. *Wrong if* the edit from 26 days ago is still having its effect.
12. **Rank 12** (pos 6.4, 99k impr, 31 clicks, edited 22 days ago). **Low**. *Wrong if* "22 days" is a bulk update date shared by thousands of pages, not a real edit of this page.
13. **Rank 13** (pos 6.5, 73k impr, 12 clicks, edited 22 days ago, 482 days old). **Low**. *Wrong if* clicks fell because of the results layout, not the snippet.
14. **Rank 14** (pos 5.9, 56k impr, 3 clicks, edited 20 days ago). **Low**. *Wrong if* 3 clicks on 56k impressions is a measurement problem.
15. **Rank 15** (pos 6.6, 84k impr, 28 clicks, 104 days since update). **Medium**. *Wrong if* the page ranks for many queries and most impressions come from ones nobody clicks.
16. **Rank 16** (pos 7.8, 66k impr, 13 clicks, edited **7 days ago**). **Low**. *Wrong if* we refresh a page that was just edited.
17. **Rank 17** (pos 7.8, 160k impr, 100 clicks, 104 days since update). **Medium**. *Wrong if* 0.06% is normal for this topic: it has the most clicks in the list, and it was not declining.
18. **Rank 18** (pos 3.7, 47k impr, 7 clicks, edited 15 days ago). **Low**. *Wrong if* position 3.7 is an average over very different queries, or the page was just edited.
19. **Rank 19** (pos 9.4, 63k impr, 21 clicks, 104 days since update). **Medium**. *Wrong if* position 9.4 is the bottom of page one, where a low CTR is expected.
20. **Rank 20** (pos 9.1, 39k impr, 4 clicks, edited **7 days ago**). **Low**. *Wrong if* we trust it: it is the smallest page in the list and was edited a week ago.

**What the review taught me.** (1) The flag works better than the ordering: flagged pages decline about 0.63 of the time versus about 0.52 for unflagged pages, but ranking by impressions puts the biggest, quietest pages first and they are not the most likely to decline. (2) Many pages share the same update gap (20, 22 or 104 days), which suggests bulk update dates, so "days since update" can mislead. (3) 13 of the 20 picks have `low` confidence, and the hand review found 8 weak picks, which is the point of this review.


## 4. Weak picks + leakage check

**Leakage check.** The score uses only three columns: `impressions_90d`, `avg_position`, `ctr`. It does **not** use `trend_direction`, `trend_pct`, `impressions_last_30d`, `impressions_prev_30d` or any other future-window or label-derived column, and no FlyRank product flags (they are not in the data). `is_declining_label` is used only for the signal tables and for precision@K.

**Honest limits.**
- `impressions_90d` and `ctr` include the last 30 days, the same days that define the label, so they overlap with the label window. The precision numbers are directional, not a forecast.
- The label ("declining") is a rule on observed impressions. A declining page may not recover after a refresh.
- Very small pages and seasonal pages can look like they are declining by chance.

In [7]:
weak = queue.head(20).query("is_declining_label == 0")[["rank", "content_id", "impressions_90d", "avg_position", "ctr", "score"]]
print(f"top-20 picks that were NOT declining: {len(weak)} of 20")
weak

top-20 picks that were NOT declining: 8 of 20


,rank,content_id,impressions_90d,avg_position,ctr,score
7678,2,content_8451fc6f034d,272144,2.3,0.03,11.7632
6903,4,content_c84a0ab98e90,223271,7.8,0.03,11.5772
3331,5,content_4a6607efcb46,128068,2.2,0.01,11.5251
3394,7,content_36ff89c8214e,295097,7.3,0.05,11.3356
4708,10,content_b115f7c74779,123469,8.0,0.03,11.0203
13631,11,content_d274ac4158ef,65138,6.8,0.01,10.8626
3070,17,content_91652435f57a,159590,7.8,0.06,10.5427
4589,18,content_339b357d04c7,46879,3.7,0.01,10.5402


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled: markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/`, then submit my repo URL on the card. Done.